**GloVe**

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from keras.preprocessing.text import Tokenizer
from sklearn.cluster import KMeans
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [ ]:
from google.colab import drive
df = pd.read_csv('/content/cleaned_Twitter_Data.csv')
print(df)

       when modi promised “minimum government maximum governance” expected him begin the difficult job reforming the state why does take years get justice state should and not business and should exit psus and temples
0       talk all the nonsense and continue all the dra...                                                                                                                                                                
1       what did just say vote for modi  welcome bjp t...                                                                                                                                                                
2       asking his supporters prefix chowkidar their n...                                                                                                                                                                
3       answer who among these the most powerful world...                                                                       

In [ ]:
max_words = 10000
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(tweets)


sequences = tokenizer.texts_to_sequences(tweets)

max_sequence_length = 100
data = pad_sequences(sequences, maxlen=max_sequence_length)
#prepares data for further processing by making each tweet a series of integers and making sure they are all uniform

In [ ]:
#loads dictionary that contains all the word to vector mappings in the GloVe embeddings file
glove_file = '/content/glove.6B.50d.txt'
embedding_dim = 50
embedding_index = {}
with open(glove_file, 'r', encoding='utf-8') as file:
    for line in file:
        values = line.split()
        word = values[0]
        coefficients = np.asarray(values[1:], dtype='float32')
        embedding_index[word] = coefficients

In [ ]:
#creating an embedding matrix using pretrained GloVe embeddings to make the tweet embeddings
word_index = tokenizer.word_index
num_words = min(max_words, len(word_index) + 1)
embedding_matrix = np.zeros((num_words, embedding_dim))
for word, i in word_index.items():
    if i >= max_words:
        continue
    embedding_vector = embedding_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(num_words, embedding_dim, weights=[embedding_matrix], input_length=max_sequence_length, trainable=False),
    tf.keras.layers.GlobalAveragePooling1D(),
])

tweet_embeddings = model.predict(data)

6007/6007 [==============================] - 11s 2ms/step


In [ ]:
num_clusters = 5
kmeans = KMeans(n_clusters=num_clusters)
cluster_labels = kmeans.fit_predict(tweet_embeddings)

df = pd.DataFrame({'tweet': tweets, 'cluster_label': cluster_labels})
# tweets clustered based on embeddings, doing 5 clusters

/usr/local/lib/python3.10/dist-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


In [ ]:
df

,tweet,cluster_label
0,when modi promised “minimum government maximum...,1
1,talk all the nonsense and continue all the dra...,4
2,what did just say vote for modi welcome bjp t...,4
3,asking his supporters prefix chowkidar their n...,1
4,answer who among these the most powerful world...,4
...,...,...
192195,why these 456 crores paid neerav modi not reco...,4
192196,dear rss terrorist payal gawar what about modi...,2
192197,did you cover her interaction forum where she ...,3
192198,there big project came into india modi dream p...,4


**BERT**

In [ ]:
from google.colab import drive
df = pd.read_csv('/content/fixed_sentiment_dataset.csv', error_bad_lines=False)


<ipython-input-7-a7482e5086dd>:2: FutureWarning: The error_bad_lines argument has been deprecated and will be removed in a future version. Use on_bad_lines in the future.


  df = pd.read_csv('/content/fixed_sentiment_dataset.csv', error_bad_lines=False)


In [ ]:
column_names = df.columns.tolist()
column_names

['0',
 'Mon Apr 06 22:19:45 PDT 2009',
 '_TheSpecialOne_',
 "@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer.  You shoulda got David Carr of Third Day to do it. ;D"]

In [ ]:
file_path = '/content/cleaned_Twitter_Data.csv'
batch_size = 1000

with open(file_path, 'r', encoding='utf-8') as file:
    tweets = file.readlines()


total_tweets = len(tweets)
for start_idx in range(0, total_tweets, batch_size):

    batch_tweets = tweets[start_idx:start_idx + batch_size]

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
encoded_tweets = tokenizer(batch_tweets, padding=True, truncation=True, return_tensors='tf')
#contains the tokenized and encoded representations of the tweets in batches, ready to be used as input to BERT

In [ ]:
model = TFBertModel.from_pretrained('bert-base-uncased')
#initialize BERT model

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.seq_relationship.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

In [ ]:
inputs = {key: encoded_tweets[key] for key in ['input_ids', 'attention_mask']}
outputs = model(inputs)

bert_embeddings = outputs.last_hidden_state

In [ ]:
bert_embeddings

<tf.Tensor: shape=(200, 74, 768), dtype=float32, numpy=
array([[[ 2.46724933e-01,  3.82029474e-01, -1.03360474e-01, ...,
         -4.29589808e-01,  3.20419014e-01,  2.07361564e-01],
        [ 7.78732240e-01,  3.94927859e-01, -9.98347476e-02, ...,
         -5.39125025e-01,  7.63382673e-01, -3.94646883e-01],
        [ 6.54251695e-01,  6.53585732e-01,  5.53894266e-02, ...,
         -2.24739760e-01,  9.12436247e-02, -3.35350364e-01],
        ...,
        [ 3.15828919e-01,  1.08228654e-01,  1.60119161e-01, ...,
          8.07666034e-02,  3.73087600e-02, -1.70948878e-02],
        [ 3.20159405e-01,  1.19729765e-01,  1.59934193e-01, ...,
          3.81266326e-02,  6.83298931e-02, -2.01723427e-02],
        [ 3.02514404e-01,  1.11452945e-01,  1.14220798e-01, ...,
         -1.52055427e-01,  8.23459923e-02, -1.38293102e-01]],

       [[-4.47289884e-01, -2.09929317e-01,  2.04284877e-01, ...,
         -5.75591683e-01,  4.46317703e-01,  3.56560290e-01],
        [ 4.84878629e-01, -5.16949058e-01, -3.5

In [ ]:
import os
import nltk
from nltk.sentiment.util import mark_negation

nltk.download('punkt')
nltk.download('vader_lexicon')

def analyze_sentiment(tweet):
    analyzer = nltk.sentiment.SentimentIntensityAnalyzer()


    tokenized_tweet = mark_negation(nltk.word_tokenize(tweet))

    sentiment_score = analyzer.polarity_scores(' '.join(tokenized_tweet))

    if sentiment_score['compound'] >= 0.05:
        sentiment_label = 'Positive'
    elif sentiment_score['compound'] <= -0.05:
        sentiment_label = 'Negative'
    else:
        sentiment_label = 'Neutral'

    return sentiment_label, sentiment_score


file_path = '/content/cleaned_Twitter_Data.csv'

limit_tweets = 1000


with open(file_path, 'r', encoding='utf-8') as file:
    tweets = file.readlines()[:limit_tweets]

for tweet in tweets:
    sentiment_label, sentiment_score = analyze_sentiment(tweet)

    print(f"Tweet: {tweet.strip()}")
    print(f"Sentiment Label: {sentiment_label}")
    print(f"Sentiment Scores: {sentiment_score}")
    print()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


Tweet: when modi promised “minimum government maximum governance” expected him begin the difficult job reforming the state why does take years get justice state should and not business and should exit psus and temples
Sentiment Label: Positive
Sentiment Scores: {'neg': 0.065, 'neu': 0.781, 'pos': 0.154, 'compound': 0.5267}

Tweet: talk all the nonsense and continue all the drama will vote for modi
Sentiment Label: Negative
Sentiment Scores: {'neg': 0.184, 'neu': 0.816, 'pos': 0.0, 'compound': -0.4019}

Tweet: what did just say vote for modi  welcome bjp told you rahul the main campaigner for modi think modi should just relax
Sentiment Label: Positive
Sentiment Scores: {'neg': 0.0, 'neu': 0.772, 'pos': 0.228, 'compound': 0.7096}

Tweet: asking his supporters prefix chowkidar their names modi did great service now there confusion what read what not now crustal clear what will crass filthy nonsensical see how most abuses are coming from chowkidars
Sentiment Label: Positive
Sentiment Score